# E-Commerce Recommendation & Fraud Detection — Analysis Notebook
**Author:** Manasa | KL University Hyderabad

Covers: 3NF schema walkthrough, ALS collaborative filtering, fraud feature analysis, ROC curve.


In [ ]:
import sys
sys.path.insert(0, '../data')
sys.path.insert(0, '../backend')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline
print('Ready')

## 1. Generate 3NF Dataset

In [ ]:
from generate_data import generate_users, generate_items, generate_interactions, generate_transactions

users = generate_users(1000)
items = generate_items(200)
interactions = generate_interactions(users, items, n=20000)
transactions = generate_transactions(users, items, n=5000)

print(f'Users:        {len(users):,}')
print(f'Items:        {len(items):,}')
print(f'Interactions: {len(interactions):,}')
print(f'Transactions: {len(transactions):,}')
print(f'Fraud rate:   {transactions["is_fraud"].mean()*100:.2f}%')

## 2. EDA — Interaction Patterns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Event distribution
event_counts = interactions['event'].value_counts()
axes[0].bar(event_counts.index, event_counts.values,
            color=['#7c4dff','#00e676','#ff6b35','#00b4ff'])
axes[0].set_title('Interaction Events')
axes[0].set_ylabel('Count')

# Interactions per user (power-law)
user_ints = interactions.groupby('user_id').size()
axes[1].hist(user_ints, bins=30, color='steelblue', edgecolor='white')
axes[1].set_title('Interactions per User (Power Law)')
axes[1].set_xlabel('Interactions')

# Category popularity
items_merged = interactions.merge(items[['item_id','category']], on='item_id')
cat_counts = items_merged['category'].value_counts()
axes[2].barh(cat_counts.index, cat_counts.values, color='mediumpurple')
axes[2].set_title('Interactions by Category')

plt.tight_layout()
plt.show()

## 3. ALS Recommender — Train & Evaluate

In [ ]:
from train_models import ALSRecommender, build_interaction_matrix

agg = interactions.groupby(['user_id','item_id'])['rating'].max().reset_index()
mat, user_ids, item_ids = build_interaction_matrix(agg)

print(f'Matrix: {mat.shape[0]} users × {mat.shape[1]} items')
print(f'Sparsity: {1 - mat.nnz / (mat.shape[0]*mat.shape[1]):.3%}')

als = ALSRecommender(n_factors=20, iterations=8)
als.fit(mat, user_ids, item_ids)

# Show recommendations for a sample user
sample_user = user_ids[0]
recs = als.recommend(sample_user, n=5)
print(f'\nTop 5 recommendations for {sample_user}:')
for item_id, score in recs:
    meta = items[items['item_id']==item_id].iloc[0] if item_id in items['item_id'].values else None
    cat = meta['category'] if meta is not None else '?'
    print(f'  {item_id} ({cat}) — score: {score:.4f}')

## 4. Fraud Detection — Feature Analysis & ROC Curve

In [ ]:
from train_models import engineer_fraud_features, train_fraud_model, FRAUD_FEATURES
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split

clf, features, metrics = train_fraud_model(transactions)
print('Model metrics:')
for k, v in metrics.items():
    if k != 'feature_importance':
        print(f'  {k}: {v}')

In [ ]:
# ROC Curve
df_eng = engineer_fraud_features(transactions)
X = df_eng[features].fillna(0)
y = df_eng['is_fraud']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
clf.fit(X_tr, y_tr)
y_prob = clf.predict_proba(X_te)[:, 1]

fpr, tpr, _ = roc_curve(y_te, y_prob)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(fpr, tpr, color='#7c4dff', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
axes[0].plot([0,1],[0,1],'k--', alpha=0.4, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Fraud Detection — ROC Curve')
axes[0].legend()

# Feature importance
fi = pd.Series(clf.feature_importances_, index=features).sort_values(ascending=True)
fi.plot(kind='barh', ax=axes[1], color='#ff6b35')
axes[1].set_title('Feature Importance — Random Forest')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()
print(f'\nFinal AUC-ROC: {roc_auc:.4f}')

## 5. Fraud Risk Factor Analysis

In [ ]:
df_eng = engineer_fraud_features(transactions)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Fraud by hour
fraud_by_hour = df_eng.groupby('hour')['is_fraud'].mean() * 100
axes[0].bar(fraud_by_hour.index, fraud_by_hour.values, color='#ff1744', alpha=0.7)
axes[0].set_title('Fraud Rate by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Fraud Rate (%)')

# Fraud by payment method
fraud_by_pay = df_eng.groupby('payment_method')['is_fraud'].mean() * 100
fraud_by_pay.sort_values().plot(kind='barh', ax=axes[1],
    color=['#00e676','#00e676','#00e676','#ffab40','#ff1744'])
axes[1].set_title('Fraud Rate by Payment Method')
axes[1].set_xlabel('Fraud Rate (%)')

# Amount distribution: fraud vs legit
fraud_amt = df_eng[df_eng['is_fraud']==1]['amount_usd'].clip(0, 800)
legit_amt = df_eng[df_eng['is_fraud']==0]['amount_usd'].clip(0, 800)
axes[2].hist(legit_amt, bins=30, alpha=0.5, label='Legitimate', color='#00e676')
axes[2].hist(fraud_amt, bins=30, alpha=0.7, label='Fraud', color='#ff1744')
axes[2].set_title('Transaction Amount: Fraud vs Legitimate')
axes[2].set_xlabel('Amount USD')
axes[2].legend()

plt.tight_layout()
plt.show()